# SpatialGDC on MOSTA
Mouse embryo spatial transcriptomics (MOSTA, 12 organs).

In [ ]:
import scanpy as sc, pandas as pd, numpy as np, sys
sys.path.insert(0, "..")
from spatialgdc import SpatialGDC, prepare_graph, compute_spatial_keep_prob, clustering, fix_seed
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

fix_seed(0)

## 1. Load and Preprocess

In [ ]:
adata = sc.read_h5ad("../dataset/Mouse_Embryo/E9.5_E1S1.MOSTA.h5ad")
adata.var_names_make_unique()

adata = adata[adata.obs["annotation"].notna()].copy()
adata.obs["Region"] = adata.obs["annotation"]
n_clusters = adata.obs["Region"].nunique()

sc.pp.filter_genes(adata, min_counts=1)
sc.pp.filter_cells(adata, min_counts=1)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=3000)
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata)
adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X.toarray())

print(f"Spots: {adata.shape[0]}, HVGs: {adata.shape[1]}, Organs: {n_clusters}")

## 2. Build Graphs and Train

In [ ]:
g_spatial = prepare_graph(adata, "spatial")
g_expr = prepare_graph(adata, "expr")

coords = adata.obsm["spatial"].copy()
coords = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
keep_prob = compute_spatial_keep_prob(g_expr, coords, sigma=0.5)

model = SpatialGDC(
    input_data=adata.obsm["X_pca"].copy(),
    graph_dict={"spatial": g_spatial, "expr": g_expr},
    n_clusters=n_clusters,
    expr_keep_prob=keep_prob,
    gamma=1.5, kappa=0.05,
    use_spatial_drop=True, use_intersection_cl=True,
)
pred_labels, embeddings, x_rec = model.train()

## 3. Evaluate

In [ ]:
adata.obsm["emb"] = embeddings
clustering(adata, n_clusters, key="emb", refinement=True, cluster_methods="mclust")

cc = "mclust_refined" if "mclust_refined" in adata.obs.columns else "mclust"
ae = adata[adata.obs.Region.notna()]
ari = adjusted_rand_score(ae.obs["Region"], ae.obs[cc])
print(f"ARI = {ari:.4f}")

## 4. Visualize

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for label in adata.obs["Region"].unique():
    mask = adata.obs["Region"] == label
    axes[0].scatter(adata.obsm["spatial"][mask,0], adata.obsm["spatial"][mask,1], s=1, label=label)
axes[0].set_title("Ground Truth"); axes[0].legend(markerscale=5, fontsize=6)
for label in adata.obs[cc].unique():
    mask = adata.obs[cc] == label
    axes[1].scatter(adata.obsm["spatial"][mask,0], adata.obsm["spatial"][mask,1], s=1)
axes[1].set_title(f"SpatialGDC (ARI={ari:.3f})")
plt.tight_layout(); plt.show()